In [56]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from googletrans import Translator
from langdetect import detect
nltk.download("stopwords")
nltk.download("vader_lexicon")



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hmodi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\hmodi\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [57]:
with open("youtube_comments.txt", "r", encoding="utf-8") as f:
    comments_list = f.readlines()

comments_list = [c.strip() for c in comments_list if c.strip()]
comments = pd.DataFrame(comments_list, columns=["comment"])

print("Total Comments Loaded:", len(comments))
comments



Total Comments Loaded: 116


,comment
0,Bahut accha video hai sir
1,This video is very helpful for beginners
2,Ye video bakwas hai kuch samajh nahi aaya
3,Amazing explanation thank you
4,Answer is option B
...,...
111,Helpful for beginners
112,Bilkul perfect explanation
113,Okay video
114,Bahut hi achi teaching


In [58]:
translator = Translator()

# common Hindi words written in English (Hinglish)
HINGLISH_WORDS = [
    "hai", "nahi", "bahut", "bilkul", "accha", "bakwas",
    "samajh", "aaya", "kya", "kaise", "kyu", "sir", "ji"
]

def translate_if_needed(text):
    try:
        lang = detect(text)

        # check for hinglish words
        is_hinglish = any(word in text.lower().split() for word in HINGLISH_WORDS)

        if lang == "hi" or is_hinglish:
            translated = translator.translate(text, dest="en")
            return translated.text
        else:
            return text

    except:
        return text

comments["translated"] = comments["comment"].apply(translate_if_needed)

comments

,comment,translated
0,Bahut accha video hai sir,very good video sir
1,This video is very helpful for beginners,This video is very helpful for beginners
2,Ye video bakwas hai kuch samajh nahi aaya,"This video is nonsense, I don't understand any..."
3,Amazing explanation thank you,Amazing explanation thank you
4,Answer is option B,Answer is option B
...,...,...
111,Helpful for beginners,Helpful for beginners
112,Bilkul perfect explanation,Bilkul perfect explanation
113,Okay video,Okay video
114,Bahut hi achi teaching,very good teaching


In [60]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   # remove links
    text = re.sub(r"[^a-z\s]", "", text)  # remove symbols
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

comments["clean_comment"] = comments["translated"].apply(clean_text)
comments

,comment,translated,clean_comment
0,Bahut accha video hai sir,very good video sir,good video sir
1,This video is very helpful for beginners,This video is very helpful for beginners,video helpful beginners
2,Ye video bakwas hai kuch samajh nahi aaya,"This video is nonsense, I don't understand any...",video nonsense dont understand anything
3,Amazing explanation thank you,Amazing explanation thank you,amazing explanation thank
4,Answer is option B,Answer is option B,answer option b
...,...,...,...
111,Helpful for beginners,Helpful for beginners,helpful beginners
112,Bilkul perfect explanation,Bilkul perfect explanation,bilkul perfect explanation
113,Okay video,Okay video,okay video
114,Bahut hi achi teaching,very good teaching,good teaching


In [62]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

comments["sentiment"] = comments["clean_comment"].apply(get_sentiment)
comments


,comment,translated,clean_comment,sentiment
0,Bahut accha video hai sir,very good video sir,good video sir,positive
1,This video is very helpful for beginners,This video is very helpful for beginners,video helpful beginners,positive
2,Ye video bakwas hai kuch samajh nahi aaya,"This video is nonsense, I don't understand any...",video nonsense dont understand anything,negative
3,Amazing explanation thank you,Amazing explanation thank you,amazing explanation thank,positive
4,Answer is option B,Answer is option B,answer option b,neutral
...,...,...,...,...
111,Helpful for beginners,Helpful for beginners,helpful beginners,positive
112,Bilkul perfect explanation,Bilkul perfect explanation,bilkul perfect explanation,positive
113,Okay video,Okay video,okay video,positive
114,Bahut hi achi teaching,very good teaching,good teaching,positive


In [65]:
sentiment_counts = comments["sentiment"].value_counts()

positive = sentiment_counts.get("positive", 0)
negative = sentiment_counts.get("negative", 0)
neutral  = sentiment_counts.get("neutral", 0)

total = positive + negative + neutral
score = (positive - negative) / total
rating = ((score + 1) / 2) * 4 + 1
rating = round(rating, 1)


In [66]:
print("\n==============================")
print("Positive:", positive)
print("Negative:", negative)
print("Neutral :", neutral)
print("Final Score:", round(score, 3))
print("⭐ Rating:", rating, "/ 5")

if score > 0.2:
    print("✅ Video is GOOD / Recommended")
elif score < 0:
    print("❌ Video is NOT GOOD")
else:
    print("⚖️ Video is AVERAGE")

print("==============================")



Positive: 62
Negative: 22
Neutral : 32
Final Score: 0.345
⭐ Rating: 3.7 / 5
✅ Video is GOOD / Recommended


In [22]:
comments

,comment,clean_comment
0,"TO Day I complete Up to <a href=""https://www.y...",day complete href
1,~No laptop guy~,laptop guy
2,10 Feb 2026 attendence,feb attendence
3,Didi laptop nahi he,didi laptop nahi
4,After 11 Feb 2016💀💀,feb
...,...,...
120,thanks alot looking up for next such helpful t...,thanks alot looking next helpful tutorials
121,"Saving my progress <br><br><a href=""https://ww...",saving progress brbra href href href href href...
122,Good video,good video
123,Didi Please share me this PDF which you showed...,didi please share pdf showed video
